## Sync vs async

In [4]:
# Tiny experiment 1 (no installs, no APIs)

In [8]:
import time

def make_chai(name):
    print(f"Boiling chai for {name}.....")
    time.sleep(3)
    print(f"Chai ready for {name}....")

start = time.time()
make_chai('Suraj')
make_chai('Swati')
make_chai('Ayesha')

print(f'Total: {time.time() - start:.1f}s ')

Boiling chai for Suraj.....
Chai ready for Suraj....
Boiling chai for Swati.....
Chai ready for Swati....
Boiling chai for Ayesha.....
Chai ready for Ayesha....
Total: 9.0s 


In [5]:
# Tiny experiment 2: the async version

In [10]:
import asyncio
import time

async def make_chai(name):
    print(f"Boiling chai for {name}.....")
    await asyncio.sleep(3)
    print(f"Chai ready for {name}....")

async def main():
    start = time.time()
    await asyncio.gather(
    make_chai('Suraj'),
    make_chai('Swati'),
    make_chai('Ayesha')
    )
    print(f"Total: {time.time() - start:.1f}s")

await main()

Boiling chai for Suraj.....
Boiling chai for Swati.....
Boiling chai for Ayesha.....
Chai ready for Suraj....
Chai ready for Swati....
Chai ready for Ayesha....
Total: 3.0s


In [21]:
import asyncio
import time

async def prepare(item, seconds):
    print(f"Started {item}")
    await asyncio.sleep(seconds)
    print(f'finished {item}')
    return f"{item} served"


async def main():
    start = time.time()
    
    result = await asyncio.gather(
    prepare('chai', 3),
    prepare('Coffee', 2),
    prepare('Samosa', 10)
    )
    print(result)
    print(f'Time {time.time() - start}')

await main()

Started chai
Started Coffee
Started Samosa
finished Coffee
finished chai
finished Samosa
['chai served', 'Coffee served', 'Samosa served']
Time 9.99428105354309


## Pydantic

In [22]:
!pip install pydantic

Defaulting to user installation because normal site-packages is not writeable


In [24]:
from pydantic import BaseModel

class ChaiOrder(BaseModel):
    name: str
    cups: int

In [31]:
order = ChaiOrder(name="Ravi", cups="3")
print(order)
print(type(order.cups))

name='Ravi' cups=3
<class 'int'>


In [27]:
order = ChaiOrder(name = "Priya", cups = "lots")
print(order)
print(order.cups)

ValidationError: 1 validation error for ChaiOrder
cups
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='lots', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [28]:
ChaiOrder(name="Amit")

ValidationError: 1 validation error for ChaiOrder
cups
  Field required [type=missing, input_value={'name': 'Amit'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [30]:
order = ChaiOrder("Ravi", 2)

print(order)
print(order.cups)

TypeError: BaseModel.__init__() takes 1 positional argument but 3 were given

In [33]:
ChaiOrder(name="Ravi", cups=2.0)   # works or fails?

ChaiOrder(name='Ravi', cups=2)

In [34]:
ChaiOrder(name="Ravi", cups=2.5)   # works or fails?

ValidationError: 1 validation error for ChaiOrder
cups
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=2.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/int_from_float

In [35]:
ChaiOrder(name=123, cups=2)        # works or fails?

ValidationError: 1 validation error for ChaiOrder
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [36]:
data = {'name': 'priya', 'cups': '4'}

order = ChaiOrder.model_validate(data)

print(order)
print(type(order.cups))

name='priya' cups=4
<class 'int'>


In [37]:
from pydantic import ValidationError

bad = {"name": "Amit", "cups": "lots"}

try:
    ChaiOrder.model_validate(bad)
except ValidationError as e:
    print("Rejected:", e.error_count(), "error")

Rejected: 1 error


#### Mini-task: async + Pydantic

In [73]:
import asyncio
import time
from pydantic import BaseModel, ValidationError

class ChaiOrder(BaseModel):
    name: str
    cups: int

raw_orders = [
    {'name': 'Ravi', 'cups': 2},
    {'name': 'Priya', 'cups': 'lots'},
    {'name': 'Amit', 'cups': '3'},
    {'name': 'Neha'},
    {"name": "Kumar", "cups": 2},       # good
    {"name": 123, "cups": "10"},     # good (text "3" becomes 3)
    {"name": "Riya", "cups": "3.5"}, # bad (not a number)
    {"name": "Rahul"},                  # bad (cups missing)
    {"cups": 2},                       # bad (name missing)
    {"name": "Sam", "cups": 2.5}      # bad (fractional)
]

In [74]:
async def process(raw):
    await asyncio.sleep(1)
    try:
        order = ChaiOrder.model_validate(raw)
        return f"Accepted: {order.name} X {order.cups}"
    except ValidationError as e:
        return f"Rejected: {raw.get('name', 'Unknown')} ({e.errors()[0]['msg']})"

In [75]:
print(await process(raw_orders[0]))

Accepted: Ravi X 2


In [76]:
async def main():
    start = time.time()
    result = await asyncio.gather(
        process(raw_orders[0]),
        process(raw_orders[1]),
        process(raw_orders[2]),
        process(raw_orders[3])
    )
    print(result)
    print(f"Total time = {time.time()-start}")

await main()

['Accepted: Ravi X 2', 'Rejected: Priya (Input should be a valid integer, unable to parse string as an integer)', 'Accepted: Amit X 3', 'Rejected: Neha (Field required)']
Total time = 1.0221788883209229


In [77]:
async def main():
    start = time.time()
    result = await asyncio.gather(*(process(r) for r in raw_orders))
    for r in result:
        print(r)
    print(f"Total time = {time.time()-start}")

await main()

Accepted: Ravi X 2
Rejected: Priya (Input should be a valid integer, unable to parse string as an integer)
Accepted: Amit X 3
Rejected: Neha (Field required)
Accepted: Kumar X 2
Rejected: 123 (Input should be a valid string)
Rejected: Riya (Input should be a valid integer, unable to parse string as an integer)
Rejected: Rahul (Field required)
Rejected: Unknown (Field required)
Rejected: Sam (Input should be a valid integer, got a number with a fractional part)
Total time = 1.0139431953430176


In [78]:
!pip install httpx

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import httpx

async with httpx.AsyncClient() as client:
    resp = await client.get("https://jsonplaceholder.typicode.com/users/1")
    print(resp.status.code())
    print(resp.jason)

In [91]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    username: str
    email: str

In [95]:
async def fetch_user(client, user_id):
    resp = await client.get(f"https://jsonplaceholder.typicode.com/users/{user_id}")
    if resp.status_code == 404:
        return f"User {user_id}: not found"
    data = resp.json()
    return User.model_validate(data)

In [96]:
import httpx

async with httpx.AsyncClient() as client:
    user = await fetch_user(client, 1)
    print(user)

id=1 name='Leanne Graham' username='Bret' email='Sincere@april.biz'


In [107]:
import asyncio
import time
import httpx

async def main():
    start = time.time()
    async with httpx.AsyncClient() as client:
        user = await asyncio.gather(*(fetch_user(client, user_id) for user_id in [*range(0,15), 999]))
    for u in user:
        if isinstance(u, str):
            print(u)
        else:
            print(f'{u.id} | {u.name} | {u.username} | {u.email}')
    print(f"Done in {time.time() - start:.2f}s")

await main()

User 0: not found
1 | Leanne Graham | Bret | Sincere@april.biz
2 | Ervin Howell | Antonette | Shanna@melissa.tv
3 | Clementine Bauch | Samantha | Nathan@yesenia.net
4 | Patricia Lebsack | Karianne | Julianne.OConner@kory.org
5 | Chelsey Dietrich | Kamren | Lucio_Hettinger@annie.ca
6 | Mrs. Dennis Schulist | Leopoldo_Corkery | Karley_Dach@jasper.info
7 | Kurtis Weissnat | Elwyn.Skiles | Telly.Hoeger@billy.biz
8 | Nicholas Runolfsdottir V | Maxime_Nienow | Sherwood@rosamond.me
9 | Glenna Reichert | Delphine | Chaim_McDermott@dana.io
10 | Clementina DuBuque | Moriah.Stanton | Rey.Padberg@karina.biz
User 11: not found
User 12: not found
User 13: not found
User 14: not found
User 999: not found
Done in 0.82s
